# draw.instant — discrete-GPU benchmarks on Colab

Runs the WebGPU kernel-fusion benchmarks on Colab's GPU. No browser — it uses **Deno's native WebGPU** (the bench files are pure WebGPU, no DOM).

**Use it:** this notebook already requests a GPU runtime, so just **Runtime → Run all**.

**Read the result:** the last cell prints the **GPU adapter first**. A `Tesla T4` (or similar) means the numbers are real. `llvmpipe` / `lavapipe` means the GPU didn't bind and you're on CPU emulation — those numbers do *not* validate the fusion thesis.

In [ ]:
# 1. Confirm a GPU runtime (Runtime → Change runtime type → GPU if this errors).
!nvidia-smi --query-gpu=name,driver_version --format=csv

In [ ]:
# 2. Install the NVIDIA Vulkan ICD whose major version MATCHES the driver.
#    libnvidia-gl-<N> ships libGLX_nvidia.so.0 + the ICD json; a mismatch fails.
import subprocess
drv = subprocess.check_output(
    "nvidia-smi --query-gpu=driver_version --format=csv,noheader",
    shell=True).decode().strip().split('.')[0]
print('driver major:', drv)
!apt-get -qq update && apt-get -qq install -y vulkan-tools libnvidia-gl-{drv} > /dev/null
# Must list the Tesla T4 — if it shows only llvmpipe, the version didn't match.
!vulkaninfo --summary 2>/dev/null | grep -iE 'deviceName|driverName' || echo 'no Vulkan device — driver mismatch'

In [ ]:
# 3. Install Deno (deno.land first; GitHub release as a fallback).
import os, glob
!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
if not glob.glob('/root/.deno/bin/deno'):
    !mkdir -p /root/.deno/bin && curl -fsSL https://github.com/denoland/deno/releases/latest/download/deno-x86_64-unknown-linux-gnu.zip -o /tmp/deno.zip && unzip -o /tmp/deno.zip -d /root/.deno/bin > /dev/null
os.environ['PATH'] = '/root/.deno/bin:' + os.environ['PATH']
!deno --version | head -1

In [ ]:
# 4. Clone the repo and run. DENO_WEBGPU_BACKEND=vulkan forces the GPU path;
#    add DENO_WEBGPU_ADAPTER_NAME=tesla if the banner still shows llvmpipe.
!git clone -q -b master https://github.com/abgnydn/draw-instant || (cd draw-instant && git pull -q)
!cd draw-instant && DENO_WEBGPU_BACKEND=vulkan deno run --unstable-webgpu -A bench-headless.mjs